# Accenture Song 2nd Round Notebook used for Demo

## 1. Data Pipeline & ETL

Build a small ETL pipeline that:
- Produces a cleaned dataset suitable for downstream use

In [1]:
import pandas as pd

CUSTOMER_DATA = "./data/customers.csv"
TRANSACTIONS_DATA = "./data/transactions.csv"

customer_raw = pd.read_csv(CUSTOMER_DATA)
transactions_raw = pd.read_csv(TRANSACTIONS_DATA)

print("---- Tabular data structure: ----")
print(customer_raw.dtypes)
print()
print(transactions_raw.dtypes)
print()
print("---- Shape of the datasets: ----")
print("customer_raw shape:", customer_raw.shape)
print("transactions_raw shape:", transactions_raw.shape)

---- Tabular data structure: ----
customer_id    int64
country          str
signup_date      str
email            str
dtype: object

transaction_id      int64
customer_id       float64
amount            float64
currency              str
timestamp             str
category              str
dtype: object

---- Shape of the datasets: ----
customer_raw shape: (5000, 4)
transactions_raw shape: (122000, 6)


In [2]:
customer_raw.head()

,customer_id,country,signup_date,email
0,1,DK,2022-01-22,user0@example.com
1,2,FI,2021-12-22,user1@example.com
2,3,SE,2023-08-18,user2@example.com
3,4,DK,2022-01-25,user3@example.com
4,5,DK,2019-09-26,user4@example.com


In [3]:
transactions_raw.head()

,transaction_id,customer_id,amount,currency,timestamp,category
0,0,1971.0,126.58,SEK,2020-04-29 06:17:00,NaN
1,1,3823.0,43.79,NOK,2020-03-28 15:14:00,food
2,2,2820.0,53.29,eur,2020-03-08 22:40:00,electronics
3,3,903.0,132.21,EUR,2020-11-23 01:02:00,unknown
4,4,365.0,75.50,SEK,2020-02-09 04:54:00,unknown


In [4]:
from etl import run_pipeline

customer_clean, transactions_clean = run_pipeline()

print("Row counts after cleaning:")
print("- customer_clean shape:", customer_clean.shape)
print("- transactions_clean shape:", transactions_clean.shape)  # dropped rows: currency NA 2644 | category NA 20220 | customer_id NA 30 

Row counts after cleaning:
- customer_clean shape: (5000, 4)
- transactions_clean shape: (97334, 6)


In [5]:
transactions_clean["currency"].value_counts()

currency
EUR    48502
SEK    24455
NOK    24377
Name: count, dtype: int64

In [6]:
transactions_clean["category"].value_counts()

category
food           32490
unknown        32441
electronics    32403
Name: count, dtype: int64

## 2. Feature Engineering + Simple Logic

From the cleaned data, implement one of the following:
- **A small feature set per customer (e.g. transaction frequency, average amount)**
- A simple rule-based classification (e.g. flagging unusual transactions)
- A lightweight model (optional, not required)

Explain:
- Why you chose these features or rules
- What business question they might support

*The selected implementation is highlighted in bold.*

In [7]:
from features import create_features

features = create_features(customer_clean, transactions_clean)
features.head()

,customer_id,country,signup_date,customer_length,total_spent,transaction_count,avg_transaction_amount,avg_monthly_spent,days_since_last_transaction,first_transaction_before_signup
0,1,DK,2022-01-22,1502,1941.31,19,102.174211,38.774501,1955,True
1,2,FI,2021-12-22,1533,1692.05,17,99.532353,33.112524,1909,True
2,3,SE,2023-08-18,929,1286.59,13,98.968462,41.547578,1940,True
3,4,DK,2022-01-25,1499,2533.56,23,110.154783,50.705003,1932,True
4,5,DK,2019-09-26,2351,1452.62,15,96.841333,18.536197,1959,False


## 3. Small LLM Pipeline

Using the text documents provided:
- Build a very small LLM-based component, such as:
  - **A simple RAG pipeline**
  - A document-based Q&A function
  - A basic inference service (local or API-based)

*The selected implementation is highlighted in bold.* 

In [8]:
from rag import load_docs, index_docs_to_db, run_rag
from huggingface_hub import InferenceClient
from dotenv import load_dotenv


load_dotenv()  # Load environment variables from .env file
docs = load_docs()
vector_db = index_docs_to_db(docs)
client = InferenceClient(model="meta-llama/Llama-3.2-1B-Instruct")
test_queries = [
    "How long does a refund take to process?",
    "When can a customer request a refund?",
    "When does a customer transaction require review?",
    "What kind of products can a customer return for a refund?",
    "What should I look out for to detect fraud?"
]

test_query = test_queries[-1]
answer = run_rag(test_query, vector_db, client)

print("👤 Query:", test_query, "\n")
print("🤖 Answer:", answer)

👤 Query: What should I look out for to detect fraud? 

🤖 Answer: To detect potential fraud, it's essential to be vigilant and monitor your transactions closely. Here are some key indicators to look out for:

1. **High transaction frequency**: If you notice a sudden increase in transactions, especially if they're all large or frequent, it could be a sign of fraudulent activity.
2. **Unusual transaction amounts**: Be cautious of transactions that are significantly larger than usual or involve unusual payment methods, such as wire transfers or gift cards.
3. **Cross-border transactions**: If you're conducting transactions with customers from other countries, be aware of potential red flags, such as transactions that involve multiple countries or currencies.
4. **Unexplained changes**: If you notice any changes in your account or transaction history that you're not aware of, it could be a sign of fraudulent activity.
5. **Suspicious login activity**: If you notice any unusual login activit

For an interactive loop with LLM, run `rag.py` separately with python.

# 4. (Optional) Serve / Output 

Optionally demonstrate how the result could be:
- **Served via a simple API**
- Written to a database or file
- Consumed by another system

*The selected implementation is highlighted in bold.*

In [12]:
!uvicorn backend:app --port 8000

INFO:     Started server process [55186]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:49858 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:49858 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:49859 - "GET /customer/1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:49940 - "GET /query-llm?what%20should%20I%20look%20out%20for%20to%20detect%20fraud HTTP/1.1" 307 Temporary Redirect
INFO:     127.0.0.1:49940 - "GET /query-llm/?what%20should%20I%20look%20out%20for%20to%20detect%20fraud HTTP/1.1" 422 Unprocessable Content
INFO:     127.0.0.1:49941 - "GET /query-llm/?q=what%20should%20I%20look%20out%20for%20to%20detect%20fraud? HTTP/1.1" 200 OK
^C
INFO:     Finished server process [55186]
ERROR:    Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.0_1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/asyncio/runners.py", 